# **Summary: Ingestion Module**

## Module Overview

This module provides a structured pipeline for importing, processing, and packaging survey data into a consistent `DataSet` object.

### Key Features

- **Data ingestion**
  - Fetches survey data from the Kobo API (`get_kobo_data`)
  - Supports local file imports (`.csv`, `.xls`, `.xlsx`) via `import_data`

- **Schema extraction (Kobo only)**
  - Builds:
    - `qdf` (questions dataframe)
    - `cdf` (choices dataframe)
  - Normalizes labels from varying Kobo formats

- **Data normalization**
  - Handles repeat groups in Kobo data
  - Flattens nested JSON into tabular format
  - Optional transformation of `select_multiple` fields into dummy variables (`split_multi`)

- **Metadata generation and logging**
  - Extracts key metadata (form ID, submission dates, country, row count)
  - Standardizes timestamps (UTC → minute precision → timezone removed)
  - Maintains an `import_log.csv` for tracking dataset imports
  - Prompts user for missing metadata when required

- **Unified dataset object**
  - Wraps all outputs into a `DataSet` class containing:
    - `df` (data)
    - `qdf` (questions, optional)
    - `cdf` (choices, optional)
    - `metadata`

- **Single entry point**
  - `build_dataset()` handles both API and file inputs and returns a ready-to-use `DataSet`

### Design Principles

- Separation of ingestion, transformation, and metadata handling
- Consistent structure across data sources
- Non-destructive transformations (methods return new dataset versions)
- Lightweight, extensible foundation for further analysis (MwM system)

# **Functions**

## Imports

In [1]:
import pandas as pd
import requests

from dotenv import load_dotenv
import os


## Core Functions

In [14]:
#================================================================
# KOBO IMPORT AND PREP FUNCTIONS
#================================================================

def get_kobo_data(BASE_URL, ASSET_ID, API_KEY):
    """
    Fetches Kobo survey data and returns a DataFrame. If GROUP_BY is specified, it will allow for repeat groups, using the name of the group.

    """

    def get_kobo_meta(BASE_URL, ASSET_ID, API_KEY):

        url = f"{BASE_URL}{ASSET_ID}/valid_content/"

        response = requests.get(
            url,
            headers={"Authorization": f"Token {API_KEY}",
                    "Accept": "application/json"}
            )
                
        def extract_label(value):
            """Return the first usable label from Kobo's variable label formats."""
            if value is None:
                return None

            if isinstance(value, str):
                cleaned = value.strip()
                return cleaned or None

            if isinstance(value, dict):
                for v in value.values():
                    if isinstance(v, str) and v.strip():
                        return v.strip()
                return None

            if isinstance(value, list):
                for v in value:
                    if isinstance(v, str) and v.strip():
                        return v.strip()
                    if isinstance(v, dict):
                        nested = extract_label(v)
                        if nested:
                            return nested
                return None

            return None

        def get_questions_df(data):
            survey = data["data"]["survey"]

            rows = []
            for q in survey:
                    rows.append({
                        "name": q.get("name"),
                        "type": q.get("type"),
                        "label": extract_label(q.get("label")),
                        "required": q.get("required", False),
                        "list_name": q.get("select_from_list_name"),
                        "relevant": q.get("relevant"),
                        "calculation": q.get("calculation")
                    })

            return pd.DataFrame(rows)

            
        def get_choices_df(data):
            choices = data["data"]["choices"]

            rows = []
            for c in choices:
                rows.append({
                    "list_name": c.get("list_name"),
                    "choice_name": c.get("name"),
                    "label": extract_label(c.get("label")),
                })

            return pd.DataFrame(rows)

        qdf = get_questions_df(response.json())
        cdf = get_choices_df(response.json())

        return qdf, cdf

    url = f"{BASE_URL}{ASSET_ID}/data/"

    qdf, cdf = get_kobo_meta(BASE_URL, ASSET_ID, API_KEY)

    if qdf['type'].str.startswith('begin_repeat').any():
        if qdf['type'].str.startswith('begin_repeat').sum() > 1:
            raise ValueError("Multiple repeat groups detected. This function currently only supports one repeat group.")
        else:
            GROUP_BY = qdf[qdf['type'].str.startswith('begin_repeat')]['name'].values[0]
    
    response = requests.get(
        url,
        headers={"Authorization": f"Token {API_KEY}",
                 "Accept": "application/json"}
    )
    
    response = response.json()

    if 'GROUP_BY' in locals():
        df = pd.json_normalize(response['results'], record_path=GROUP_BY, meta = [elem for elem in response['results'][0].keys() if elem != GROUP_BY])
        df.columns = df.columns.str.replace(f"{GROUP_BY}/", '', regex=False)

    else:
        df = pd.json_normalize(response['results'])
    
    form_id = df['_xform_id_string'].unique()

    
    if len(form_id) > 1:
        raise ValueError("Multiple form IDs detected. This function currently only supports one form ID.")
    
    qdf['form_id'] = form_id[0]
    cdf['form_id'] = form_id[0]
    df = df.rename(columns = {'_xform_id_string': 'form_id'})

    return df, qdf, cdf

def import_data(file_path):

    if file_path.endswith('.xls') or file_path.endswith('.xlsx'):
        df = pd.read_excel(file_path)
    elif file_path.endswith('.csv'):
        for sep in [',', ';']:
                try:
                    df = pd.read_csv(file_path, sep=sep)
                    # basic sanity check: more than 1 column
                    if df.shape[1] > 1:
                        return df
                except Exception:
                    pass
        raise ValueError("Could not detect delimiter")
    else:
        raise ValueError("Unsupported file format. Please provide a .csv, .xls, or .xlsx file.")    
    
    return df

def split_multi(df, qdf):

    if not qdf['type'].str.startswith('select_multiple').any():
        return df
    
    columns =  qdf[qdf['type'].str.startswith('select_multiple')]['name'].values
    
    # split into columns
    for col in columns:
        dummies = df[col].str.get_dummies(sep=" ")
        dummies.columns = [f"{col}_{subcol}" for subcol in dummies.columns]
        df = pd.concat([df, dummies], axis=1)
        df.drop(columns = [col], inplace=True)
    return df

def add_to_log(df, source = 'kobo'):

    import os
    import json
    from pathlib import Path

    def find_project_root(start_path: Path, marker_file: str = 'path_config.json') -> Path:
    
        for candidate in [start_path.resolve(), *start_path.resolve().parents]:
            if (candidate / marker_file).exists():
                return candidate
        raise FileNotFoundError(f'Could not find {marker_file} above {start_path}')

    PROJECT_ROOT = find_project_root(Path.cwd())
    CONFIG_PATH = PROJECT_ROOT / "path_config.json"

    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        PATHS = json.load(f)

    DATA_DIR = (PROJECT_ROOT / PATHS["data_dir"]).resolve()

    log_path = f"{DATA_DIR}/import_log.csv"

    if not os.path.exists(log_path):
        log_df = pd.DataFrame(columns=['form_id', 'survey_name', 'first_submission', 'last_submission', 'rows', 'country', '_uuid_cols','logged_at'])
    else:
        log_df = pd.read_csv(log_path)
    
    def parse_kobo_metadata(df):

        form_id = df['form_id'].unique()[0]
        survey_name = None
        
        df['_submission_time'] = pd.to_datetime(df['_submission_time'], utc=True, errors='coerce').dt.floor('min').dt.tz_localize(None)
        
        first_submission = df['_submission_time'].min().floor('min')
        last_submission = df['_submission_time'].max().floor('min')
        rows = len(df)

        df_low = df.copy()
        df_low.columns = df_low.columns.str.lower()

        if 'country' in df_low.columns:
            if df_low['country'].nunique() > 1:
                country = 'multiple'
            else:
                country = df_low['country'].unique()[0]
        else:
            country = None

        metadata = {
            'form_id': form_id,
            'survey_name': survey_name,
            'first_submission': first_submission,
            'last_submission': last_submission,
            'rows': rows,
            'country': country,
            '_uuid_cols': None
        }

        return metadata
    
    def parse_xls_csv_metadata(df):

        if 'form_id' in df.columns:
            form_id = df['form_id'].unique()[0]
        else:
            form_id = None
        
        survey_name = None
        if '_submission_time' not in df.columns:
            
            cols = df.columns.str.lower()

            if any(col in cols for col in ['date', 'submission_date', 'timestamp']):
                date_col = next(col for col in df.columns if col.lower() in ['date', 'submission_date', 'timestamp'])
                df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
                df['_submission_time'] = df[date_col]

            else:
                df['_submission_time'] = pd.NaT

        df['_submission_time'] = pd.to_datetime(df['_submission_time'], utc=True, errors='coerce').dt.floor('min').dt.tz_localize(None)

        first_submission = pd.to_datetime(df['_submission_time'].min(), errors='coerce').floor('min')
        last_submission = pd.to_datetime(df['_submission_time'].max(), errors='coerce').floor('min')
        
        rows = len(df)

        df_low = df.copy()
        df_low.columns = df_low.columns.str.lower()

        if 'country' in df_low.columns:
            if df_low['country'].nunique() > 1:
                country = 'multiple'
            else:
                country = df_low['country'].unique()[0]
        else:
            country = None

        metadata = {
            'form_id': form_id,
            'survey_name': survey_name,
            'first_submission': first_submission,
            'last_submission': last_submission,
            'rows': rows,
            'country': country,
            '_uuid_cols': None
        }

        return metadata

    if source == 'kobo':
        metadata = parse_kobo_metadata(df)
    else:
        metadata = parse_xls_csv_metadata(df)
    
    if metadata['form_id'] is None:
        metadata['form_id'] = input("No form ID found in the data. Please enter a form ID to use for logging purposes. if this form has been used previously, use the same form ID to link the datasets. Form ID: ")

    if metadata['form_id'] in log_df['form_id'].values:
        print(f"Form ID {metadata['form_id']} already exists in the log. Processed data will be added to the existing dataset.")

        metadata['survey_name'] = log_df[log_df['form_id'] == metadata['form_id']]['survey_name'].values[0]
        metadata['country'] = log_df[log_df['form_id'] == metadata['form_id']]['country'].values[0]
        metadata['first_submission'] = log_df[log_df['form_id'] == metadata['form_id']]['first_submission'].values[0]
        metadata['last_submission'] = log_df[log_df['form_id'] == metadata['form_id']]['last_submission'].values[0]
        metadata['_uuid_cols'] = log_df[log_df['form_id'] == metadata['form_id']]['_uuid_cols'].values[0]

    else:
        print(f"Form ID {metadata['form_id']} added to log. Processed data will be saved as a new file.")

        metadata['survey_name'] = input("Please enter the survey name: ")

        if metadata['country'] is None:
            metadata['country'] = input("Please enter the country name for this survey (multiple if applicable): ")
        
        if metadata['first_submission'] is pd.NaT:
            date_input = input("No submission time found. Please enter the date of the first submission (YYYY-MM-DD) or leave blank: ")
            if date_input:
                metadata['first_submission'] = pd.to_datetime(date_input, errors='coerce')
            else:
                metadata['first_submission'] = pd.NaT
        
        if metadata['last_submission'] is pd.NaT:
            date_input = input("No submission time found. Please enter the date of the last submission (YYYY-MM-DD) or leave blank: ")
            if date_input:
                if metadata['first_submission'] is not pd.NaT and pd.to_datetime(date_input, errors='coerce') < metadata['first_submission']:
                    print("Last submission date cannot be before first submission date. Please enter a valid date.")
                    date_input = input("Please enter the date of the last submission (YYYY-MM-DD) or leave blank: ")
                metadata['last_submission'] = pd.to_datetime(date_input, errors='coerce')
            else:
                metadata['last_submission'] = pd.NaT

    log_df = pd.concat([log_df, pd.DataFrame([metadata])], ignore_index=True)
    
    log_df['logged_at'] = pd.to_datetime(log_df['logged_at'], errors='coerce')

    log_df.loc[log_df.index[-1], 'logged_at'] = pd.Timestamp.now().floor('min')

    log_df.to_csv(log_path, index=False)

    print("Metadata added to log:")
    print(metadata)

    return metadata, log_df

class DataSet:
    def __init__(self, df, qdf, cdf, metadata):
        self.df = df
        self.qdf = qdf
        self.cdf = cdf
        self.metadata = metadata

    def split_multi(self):
        df = self.df.copy()

        if not self.qdf['type'].str.startswith('select_multiple').any():
            return DataSet(df, self.qdf, self.cdf, self.metadata)
        
        columns =  self.qdf[self.qdf['type'].str.startswith('select_multiple')]['name'].values
    
    # split into columns
        for col in columns:
            dummies = df[col].str.get_dummies(sep=" ")
            dummies.columns = [f"{col}_{subcol}" for subcol in dummies.columns]
            df = pd.concat([df, dummies], axis=1)
            df.drop(columns = [col], inplace=True)

        return DataSet(df, self.qdf, self.cdf, self.metadata)
    
def add_uid(df):
    if ['name', 'child_name', 'parent_name', 'student_name', 'teacher_name'] in df.columns.str.lower.str.replace(" ", "_"): 
        if 'uid' not in df.columns:
            print("")
            df['uid'] = pd.util.hash_pandas_object(df, index=False).astype(str)

def add_uuid(dataset, hash_cols):

    import hashlib

    def find_hash_columns(dataset):
        cols = dataset.df.columns.tolist()

        for i in range(1, len(cols) + 1):
            subset = cols[:i]
            if dataset.df.duplicated(subset=subset).sum() == 0:
                return subset

        # fallback: all columns (still may not be unique!)
        return cols

    def make_uuid(row, hash_cols):
        key = "|".join(str(row[col]) for col in hash_cols)
        return hashlib.md5(key.encode()).hexdigest()
    
    if '_uuid' in dataset.df.columns:
        print("UUID column already exists. Skipping UUID generation.")
        return dataset
    
    elif dataset.metadata['_uuid_cols'] is not None:
        hash_cols = dataset.metadata['_uuid_cols'].split(",")
        print(f"Using previously logged hash columns for UUID generation: {hash_cols}")

    else:
        hash_cols = find_hash_columns(dataset.df)

    if len(hash_cols) > 5:
        print("UID depends on many columns – may be unstable")

    dataset.df["_uuid"] = dataset.df.apply(
        lambda row: make_uuid(row, hash_cols),
        axis=1
        )

    dataset.df["_uuid_cols"] = ",".join(hash_cols)

    return dataset

def build_dataset(BASE_URL=None, ASSET_ID=None, API_KEY=None, file_path=None):

    # If file_path is provided, import data from file. Otherwise, fetch data from Kobo API.
    if file_path:
        df = import_data(file_path)
        metadata, _ = add_to_log(df, source='file')
        df['_submission_time'] = df['_submission_time'].fillna(metadata['last_submission'])
        print("Data imported from file. Kobo metadata functions will be skipped (question dataframe and choices dataframe). Only basic metadata will be logged.")
        dataset = DataSet(df, None, None, metadata)
        dataset = add_uuid(dataset, hash_cols=None)
        return dataset
    
    if not all([BASE_URL, ASSET_ID, API_KEY]):
        raise ValueError("BASE_URL, ASSET_ID, and API_KEY must be provided if file_path is not specified.")
    
    df, qdf, cdf = get_kobo_data(BASE_URL, ASSET_ID, API_KEY)
    metadata, _ = add_to_log(df, source='kobo')
    print("Data fetched from Kobo API and metadata logged including question and choice dataframes (.qdf and .cdf).")
    dataset = DataSet(df, qdf, cdf, metadata)
    dataset = add_uuid(dataset, hash_cols=None)
    return dataset


# **Aggregate**

In [3]:
def aggregates(df):

    from pathlib import Path
    import json

    def find_project_root(start_path: Path, marker_file: str = 'path_config.json') -> Path:
    
        for candidate in [start_path.resolve(), *start_path.resolve().parents]:
            if (candidate / marker_file).exists():
                return candidate
        raise FileNotFoundError(f'Could not find {marker_file} above {start_path}')
        
    PROJECT_ROOT = find_project_root(Path.cwd())
    CONFIG_PATH = PROJECT_ROOT / "path_config.json"

    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        PATHS = json.load(f)

    CONFIG_DIR = (PROJECT_ROOT / PATHS["config_dir"]).resolve()

    def calculate_dims(df):

        # ---------- helpers ----------

        def resolve_column(df, candidates):
            """Return first matching column in df"""
            for col in candidates:
                if col in df.columns:
                    return col
            return None

        def bin_values(series, bins):
            """Apply binning logic"""
            def assign(val):
                if pd.isna(val):
                    return None
                
                for b in bins:
                    if b["min"] <= val <= b["max"]:
                        return b["label"]
                
                return None

            return series.apply(assign)

        def compute_derived_dimension(series, dim):
            """Compute derived dimension"""
            if dim["operation"] == "bin":
                return bin_values(series, dim["bins"])
            return None

        # ---------- build dimension map ----------
               
        with open(CONFIG_DIR / "dimensions_config.json", "r", encoding="utf-8") as f:
            dimensions_config = json.load(f)

        dimension_map = {
            dim["id"]: dim for dim in dimensions_config["dimensions"]
        }

        # ---------- resolve columns ----------

        resolved_columns = {}

        for dim_id, dim in dimension_map.items():

            if dim["type"] in ["categorical", "numeric"]:
                resolved_columns[dim_id] = resolve_column(df, dim["columns"])

            elif dim["type"] == "derived":
                resolved_columns[dim_id] = resolve_column(df, dim["columns"])

        # ---------- determine availability ----------

        available_dimensions = []
        missing_dimensions = []

        for dim_id, col in resolved_columns.items():
            if col is not None:
                available_dimensions.append(dim_id)
            else:
                missing_dimensions.append(dim_id)

        # ---------- compute / map dimensions ----------

        new_cols = {}
        dims_derived = []

        for dim_id in available_dimensions:
            dim = dimension_map[dim_id]
            col = resolved_columns[dim_id]

            if dim["type"] == "derived":
                new_cols[dim_id] = compute_derived_dimension(df[col], dim)
                dims_derived.append(dim_id)

            elif dim["type"] in ["categorical", "numeric"]:
                if col != dim_id:  # only copy if needed
                    new_cols[dim_id] = df[col]

        if new_cols:
            df = pd.concat([df, pd.DataFrame(new_cols)], axis=1)

        # ---------- logging ----------

        print("Available dimensions:", available_dimensions)
        print("Missing dimensions:", missing_dimensions)
        print("Resolved columns:", resolved_columns)
        print("Derived dimensions computed:", dims_derived)

        return df, available_dimensions, missing_dimensions

    def calculate_indicators(df):

        with open(CONFIG_DIR / "child_config.json", "r", encoding="utf-8") as f:
            child_config = json.load(f)

        indicator_map = {
            ind["id"]: ind for ind in child_config["indicators"]
        }

        def resolve_column(df, candidates):
            cols = [col for col in candidates if col in df.columns]
            return cols if cols else None

        available_indicators = []
        missing_indicators = []
        resolved_columns = {}

        # --- Resolve calculated indicators ---
        for ind_id, ind in indicator_map.items():
            if ind["agg_type"] == "calculated":
                requested_cols = ind["calc"]["columns"]
                found_cols = resolve_column(df, requested_cols)

                resolved_columns[ind_id] = found_cols

                if found_cols is not None and len(found_cols) == len(requested_cols):
                    available_indicators.append(ind_id)
                else:
                    missing_indicators.append(ind_id)

        # --- Compute calculated indicators (vectorised) ---
        ind_df = pd.DataFrame(index=df.index)

        for ind in available_indicators:
            meta = indicator_map[ind]
            op = meta["calc"]["op"]
            cols = resolved_columns[ind]

            if op == "sum":
                ind_df[ind] = df[cols].sum(axis=1)

            elif op == "mean":
                ind_df[ind] = df[cols].mean(axis=1)

            elif op == "divide":
                denom = meta["calc"]["by"]
                ind_df[ind] = df[cols].sum(axis=1) / denom

            elif op == "max":
                # max index where value == 1
                mask = df[cols].eq(1)

                weights = pd.DataFrame(
                    [range(1, len(cols)+1)] * len(df),
                    columns=cols,
                    index=df.index
                )

                ind_df[ind] = (mask * weights).max(axis=1)

        # --- Resolve aggregate indicators (based on calculated outputs) ---
        agg_inds = []

        for ind_id, ind in indicator_map.items():
            if ind["agg_type"] == "aggregate":
                requested_cols = ind["calc"]["columns"]
                found_cols = resolve_column(ind_df, requested_cols)

                resolved_columns[ind_id] = found_cols

                if found_cols is not None and len(found_cols) == len(requested_cols):
                    agg_inds.append(ind_id)
                    available_indicators.append(ind_id)
                else:
                    missing_indicators.append(ind_id)

        # --- Compute aggregates (vectorised) ---
        agg_df = pd.DataFrame(index=df.index)

        for ind in agg_inds:
            cols = resolved_columns[ind]
            agg_df[ind] = ind_df[cols].mean(axis=1)

        # --- Combine ---
        df = pd.concat([df, ind_df, agg_df], axis=1)

        print("Available indicators:", available_indicators)
        print("Missing indicators:", missing_indicators)

        return df, available_indicators, missing_indicators

    df, available_dimensions, missing_dimensions = calculate_dims(df)
    df, available_indicators, missing_indicators = calculate_indicators(df)

    df_summary = {
        "available_dimensions": available_dimensions,
        "missing_dimensions": missing_dimensions,
        "available_indicators": available_indicators,
        "missing_indicators": missing_indicators
    }

    df_short = df[available_dimensions + available_indicators]

    return df, df_short, df_summary

## Test and Notes

In [4]:
load_dotenv()

BASE_URL = os.getenv("KOBO_BASE_URL")
ASSET_ID = "aDvnrQxAnvFdrpBBunG97Y"
API_KEY = os.getenv("KOBO_API_KEY")

In [15]:
dataset = build_dataset(BASE_URL=BASE_URL, ASSET_ID=ASSET_ID, API_KEY=API_KEY)
dataset.df, dataset.df_short, dataset.df_summary = aggregates(dataset.df)

Form ID aDvnrQxAnvFdrpBBunG97Y already exists in the log. Processed data will be added to the existing dataset.
Metadata added to log:
{'form_id': 'aDvnrQxAnvFdrpBBunG97Y', 'survey_name': np.int64(238729), 'first_submission': '2026-05-22 15:19:00', 'last_submission': '2026-06-07 11:48:00', 'rows': 29, 'country': 'multiple', '_uuid_cols': np.float64(nan)}
Data fetched from Kobo API and metadata logged including question and choice dataframes (.qdf and .cdf).
UUID column already exists. Skipping UUID generation.
Available dimensions: ['admin0']
Missing dimensions: ['admin1', 'admin2', 'admin3', 'location', 'protection_status', 'school', 'sex', 'age', 'disability', 'grade', 'age_group']
Resolved columns: {'admin0': 'country', 'admin1': None, 'admin2': None, 'admin3': None, 'location': None, 'protection_status': None, 'school': None, 'sex': None, 'age': None, 'disability': None, 'grade': None, 'age_group': None}
Derived dimensions computed: []
Available indicators: []
Missing indicators: [

In [ ]:
dataset = build_dataset(file_path=r"../data/aser_sample.csv")
dataset.df, dataset.df_inds, dataset.inds_summary = aggregates(dataset.df)

Form ID aser_01 added to log. Processed data will be saved as a new file.
Metadata added to log:
{'form_id': 'aser_01', 'survey_name': 'aser_sample', 'first_submission': Timestamp('2022-01-02 00:00:00'), 'last_submission': Timestamp('2022-03-04 00:00:00'), 'rows': 200, 'country': 'Kenya'}
Data imported from file. Kobo metadata functions will be skipped (question dataframe and choices dataframe). Only basic metadata will be logged.
Available dimensions: ['admin1', 'location', 'protection_status', 'school', 'sex', 'age', 'disability', 'age_group']
Missing dimensions: ['admin0', 'admin2', 'admin3', 'grade']
Resolved columns: {'admin0': None, 'admin1': 'region', 'admin2': None, 'admin3': None, 'location': 'location', 'protection_status': 'protection_status', 'school': 'school_name', 'sex': 'child_sex', 'age': 'child_age', 'disability': 'disability_status', 'grade': None, 'age_group': 'child_age'}
Derived dimensions computed: ['age_group']
Available indicators: ['aser_literacy', 'aser_numer

In [ ]:

import pickle

with open("../data/datasets/aser_test.pkl", "wb") as f:
    pickle.dump(dataset, f)


**notes**

- add in function to check inputted country names against iso3 list, and add iso3 name as well. Return error if name doesn't match. 
- do we want to have split multi built in automatically? or would we preserve this as an option. Currently produces new object - could be in place transformation.



**testing**

- Kobo form: used a GPE survey which includes repeat groups and select multiples
- Kobo excel output: used "MwM Proof of Concept\data\demo_data\Formative_Assessment_DummyData.xlsx"
- Random csv: used a table output from the GPE survey with no date or relevant metadata


